In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U torchao

In [ ]:
!pip install -U transformers peft datasets accelerate bitsandbytes

In [ ]:
!unzip -q "/content/drive/MyDrive/dataset.zip" -d "/content/"

In [ ]:
import pandas as pd
import json
import os

csv_path = "/content/generalization_test/labels.csv"
output_file = "/content/generalization_vqa.jsonl"

image_folder = "/content/generalization_test/images" # Το πιο πιθανό σενάριο
if not os.path.exists(image_folder):
    for root, dirs, files in os.walk("/content/"):
        if 'images' in dirs and 'sample_data' not in root:
            image_folder = os.path.join(root, 'images')
            break

print(f"CSV Path: {csv_path}")
print(f"Φάκελος Εικόνων: {image_folder}")

if not os.path.exists(csv_path):
    print("Σφάλμα: Το labels.csv δεν βρέθηκε στο /content/")
else:
    # 1. Φόρτωση CSV
    print("\n Φόρτωση του CSV")
    with open(csv_path, 'r', encoding='utf-8') as f:
        first_line = f.readline().strip()


    if first_line == 'labels' or ';' not in first_line:
        df = pd.read_csv(csv_path, sep=',' if ',' in first_line else ';', skiprows=0 if ',' in first_line else 1)
    else:
        df = pd.read_csv(csv_path, sep=';')

    df.columns = df.columns.str.strip()

    vqa_data = []
    found_images = 0
    missing_images = 0

    print("Μετατροπή σε VQA Dataset...")

    for index, row in df.iterrows():
        img_path = os.path.join(image_folder, str(row['filename']))

        if not os.path.exists(img_path):
            missing_images += 1
            continue

        found_images += 1
        vqa_data.append({"image": img_path, "question": "What is the modulation scheme of this constellation?", "answer": str(row['modulation'])})
        vqa_data.append({"image": img_path, "question": "Identify the level of phase noise in this signal.", "answer": str(row['phase_noise'])})
        vqa_data.append({"image": img_path, "question": "What is the severity of the I/Q imbalance?", "answer": str(row['iq_imbalance'])})
        vqa_data.append({"image": img_path, "question": "Is there jamming present in this constellation? Describe it.", "answer": str(row['jamming'])})
        vqa_data.append({"image": img_path, "question": "Classify the Signal-to-Noise Ratio (SNR) range of this signal.", "answer": str(row['snr_range'])})

    with open(output_file, 'w', encoding='utf-8') as f:
        for item in vqa_data:
            f.write(json.dumps(item) + '\n')

    print(f"\n ΟΛΟΚΛΗΡΩΘΗΚΕ Βρέθηκαν {found_images} εικόνες (Αγνοήθηκαν {missing_images} που έλειπαν).")
    print(f"Δημιουργήθηκαν συνολικά {len(vqa_data)} δείγματα ερωταπαντήσεων στο {output_file}.")

In [ ]:
import pandas as pd
import json
import os
import random

csv_path = "/content/labels.csv"
image_folder = "/content/images"

df = pd.read_csv(csv_path)

df.columns = df.columns.str.strip()

print("Επιτυχής Φόρτωση CSV. Οι στήλες είναι:")
print(df.columns.tolist())
print("-" * 40)

vqa_data = []

for index, row in df.iterrows():
    img_path = os.path.join(image_folder, str(row['filename']))

    # 1. Modulation
    vqa_data.append({
        "image": img_path,
        "question": "What is the modulation scheme of this constellation?",
        "answer": str(row['modulation'])
    })

    # 2. Phase Noise
    vqa_data.append({
        "image": img_path,
        "question": "Identify the level of phase noise in this signal.",
        "answer": str(row['phase_noise'])
    })

    # 3. I/Q Imbalance
    vqa_data.append({
        "image": img_path,
        "question": "What is the severity of the I/Q imbalance?",
        "answer": str(row['iq_imbalance'])
    })

    # 4. Jamming
    vqa_data.append({
        "image": img_path,
        "question": "Is there jamming present in this constellation? Describe it.",
        "answer": str(row['jamming'])
    })

    # 5. SNR Range
    vqa_data.append({
        "image": img_path,
        "question": "Classify the Signal-to-Noise Ratio (SNR) range of this signal.",
        "answer": str(row['snr_range'])
    })

random.shuffle(vqa_data)

output_file = "/content/rf_vqa_dataset.jsonl"
with open(output_file, 'w', encoding='utf-8') as f:
    for item in vqa_data:
        f.write(json.dumps(item) + '\n')

print(f"\nΔημιουργήθηκαν {len(vqa_data)} δείγματα ερωταπαντήσεων.")

In [ ]:
from datasets import load_dataset
from transformers import AutoProcessor

model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"
processor = AutoProcessor.from_pretrained(model_id)

print("Φόρτωση του dataset...")
dataset = load_dataset("json", data_files="/content/rf_vqa_dataset.jsonl", split="train")

SYSTEM_MESSAGE = "You are an expert AI assistant specialized in Radio Frequency (RF) signal analysis and wireless communications. Your task is to analyze constellation diagrams and accurately identify their characteristics."

def format_and_prepare_dataset(example):
    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_MESSAGE}]
        },
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": example["question"]}
            ]
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": example["answer"]}]
        }
    ]

    text = processor.apply_chat_template(messages, add_generation_prompt=False)

    return {
        "text": text,
        "image_path": example["image"]
    }

print("Εφαρμογή του Chat Template σε 163.840 δείγματα")
processed_dataset = dataset.map(format_and_prepare_dataset, remove_columns=dataset.column_names)

print("\n" + "="*50)
print(" ΠΑΡΑΔΕΙΓΜΑ ΕΝΟΣ ΕΤΟΙΜΟΥ ΔΕΙΓΜΑΤΟΣ")
print("="*50)
print(processed_dataset[0]['text'])

In [ ]:
import random

print("1. Εντοπισμός όλων των μοναδικών εικόνων...")
all_unique_images = list(set(processed_dataset["image_path"]))
random.shuffle(all_unique_images)

total_images = len(all_unique_images)
print(f"Βρέθηκαν {total_images} μοναδικές εικόνες.")

train_split = int(0.80 * total_images)
val_split = int(0.90 * total_images)

train_images = set(all_unique_images[:train_split])
val_images = set(all_unique_images[train_split:val_split])
test_images = set(all_unique_images[val_split:])

print(f"Χωρισμός: {len(train_images)} Train | {len(val_images)} Val | {len(test_images)} Test εικόνες.")

print("\nΔημιουργία των τελικών splits...")
train_dataset = processed_dataset.filter(lambda x: x["image_path"] in train_images)
val_dataset = processed_dataset.filter(lambda x: x["image_path"] in val_images)
test_dataset = processed_dataset.filter(lambda x: x["image_path"] in test_images)

print("\n" + "="*50)
print(" ΤΕΛΙΚΟΣ ΟΓΚΟΣ ΔΕΔΟΜΕΝΩΝ (Ερωταπαντήσεις)")
print("="*50)
print(f"Train Set: {len(train_dataset)} δείγματα")
print(f"Validation Set: {len(val_dataset)} δείγματα")
print(f"Test Set: {len(test_dataset)} δείγματα (Κράτα τα για το τέλος!)")

In [ ]:
import torch
import random
import json
from PIL import Image
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForImageTextToText, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

# ── 1. Processor ───────────────────────────────────────────
model_id  = "HuggingFaceTB/SmolVLM-256M-Instruct"
processor = AutoProcessor.from_pretrained(model_id)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# ── 2. Dataset Format ──────────────────────────────────────
dataset = load_dataset("json",
                       data_files="/content/rf_vqa_dataset.jsonl",
                       split="train")

SYSTEM = "You are an RF expert. Answer with one word only. No explanation."

def format_func(example):
    messages = [
        {"role": "system",    "content": [{"type": "text",  "text": SYSTEM}]},
        {"role": "user",      "content": [{"type": "image"},
                                          {"type": "text", "text": example["question"]}]},
        {"role": "assistant", "content": [{"type": "text",  "text": example["answer"]}]}
    ]
    return {
        "text":       processor.apply_chat_template(messages, add_generation_prompt=False),
        "image_path": example["image"]
    }

processed_dataset = dataset.map(format_func, remove_columns=dataset.column_names)
print(f"Total samples: {len(processed_dataset)}")

# ── 3. Split ───────────────────────────────────────────────
all_images = list(set(processed_dataset["image_path"]))
random.seed(42)
random.shuffle(all_images)
subset_images = set(all_images[:12000])
mini_dataset  = processed_dataset.filter(
    lambda x: x["image_path"] in subset_images)

train_split = int(0.80 * 12000)
val_split   = int(0.90 * 12000)
train_imgs  = set(all_images[:train_split])
val_imgs    = set(all_images[train_split:val_split])
test_imgs   = set(all_images[val_split:])

train_dataset = mini_dataset.filter(lambda x: x["image_path"] in train_imgs)
val_dataset   = mini_dataset.filter(lambda x: x["image_path"] in val_imgs)
test_dataset  = mini_dataset.filter(lambda x: x["image_path"] in test_imgs)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ── 4. Model + LoRA ────────────────────────────────────────
model = AutoModelForImageTextToText.from_pretrained(
    model_id, device_map="auto", torch_dtype=torch.bfloat16
)
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM"
)
peft_model = get_peft_model(model, lora_config)
peft_model.gradient_checkpointing_enable()
peft_model.config.use_cache = False
peft_model.print_trainable_parameters()

# ── 5. Collator ────────────────────────────────────────────
class VLMDataCollator:
    def __init__(self, processor):
        self.processor = processor
        self.assistant_token_ids = processor.tokenizer.encode(
            "Assistant", add_special_tokens=False
        )  # [9519, 9531]

    def __call__(self, examples):
        texts  = [e["text"] for e in examples]
        images = [Image.open(e["image_path"]).convert("RGB")
                  for e in examples]

        batch  = self.processor(
            text=texts, images=images,
            return_tensors="pt", padding=True
        )
        labels = batch["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        for i in range(len(examples)):
            input_ids    = batch["input_ids"][i].tolist()
            answer_start = self._find_answer_start(input_ids)
            if answer_start is not None:
                labels[i, :answer_start] = -100
            else:
                labels[i, :] = -100
                print(f" Pattern not found for sample {i}")

        batch["labels"] = labels
        return batch

    def _find_answer_start(self, input_ids):
        pattern = self.assistant_token_ids + [42]  # 'Assistant:'
        for idx in range(len(input_ids) - len(pattern)):
            if input_ids[idx:idx+len(pattern)] == pattern:
                return idx + len(pattern)  #  χωρίς +1
        return None


from transformers import TrainerCallback

class PrintLossCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 10 == 0 and len(state.log_history) > 0:
            # Βρες το τελευταίο log entry που έχει loss
            for log in reversed(state.log_history):
                if "loss" in log:
                    print(f"Step {state.global_step:4d} | "
                          f"Loss: {log['loss']:.4f} | "
                          f"LR: {log.get('learning_rate', 0):.2e} | "
                          f"Epoch: {state.epoch:.3f}")
                    break

# ── 6. Training Arguments ──────────────────────────────────
training_args = TrainingArguments(
    output_dir="/content/smolvlm_fixed",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,    # effective batch = 32
    learning_rate=1e-4,
    num_train_epochs=2,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    optim="adamw_torch",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=8,
    warmup_ratio=0.05,
)

# ── 7. Trainer ─────────────────────────────────────────────
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=VLMDataCollator(processor),
)

print("\nΞεκινάει το training...")
trainer.train()


# ── 8. Save ────────────────────────────────────────────────
peft_model.save_pretrained("/content/smolvlm_rf_final")
processor.save_pretrained("/content/smolvlm_rf_final")
print(" Μοντέλο αποθηκεύτηκε!")

In [ ]:
import os
from google.colab import drive



drive_save_path = '/content/drive/MyDrive/MyTrainedModels/smolvlm_rf_final'

os.makedirs(drive_save_path, exist_ok=True)

print(f"Ξεκινάει η αποθήκευση του μοντέλου στο: {drive_save_path} ...")

peft_model.save_pretrained(drive_save_path)
processor.save_pretrained(drive_save_path)

print(" Το μοντέλο και ο processor αποθηκεύτηκαν επιτυχώς στο Google Drive")

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

model_folder = "smolvlm_rf_final"
drive_path = "/content/drive/MyDrive/MyTrainedModels/"


if not os.path.exists(drive_path):
    os.makedirs(drive_path)

shutil.make_archive(model_folder, 'zip', model_folder)

print("Μεταφορά στο Google Drive...")
shutil.copy(f"{model_folder}.zip", drive_path)

print(f"Το μοντέλο αποθηκεύτηκε επιτυχώς στο: {drive_path}{model_folder}.zip")

In [ ]:
import torch
import json
import random
import os
import zipfile
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel  # <-- Προσθήκη για φόρτωση finetuned μοντέλου
from collections import defaultdict
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

# ── 0. Βελτιστοποιήσεις Ταχύτητας για A100 ─────────────────
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ── 1. Αποσυμπίεση και Φόρτωση FINETUNED Μοντέλου ──────────
zip_path = "/content/drive/MyDrive/MyTrainedModels/smolvlm_rf_final.zip"
extract_path = "/content/smolvlm_rf_final"

if not os.path.exists(extract_path):
    print(f"Εξαγωγή του μοντέλου από {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(" Η εξαγωγή ολοκληρώθηκε!")

model_id   = "HuggingFaceTB/SmolVLM-256M-Instruct"
processor  = AutoProcessor.from_pretrained(model_id)

print("Φόρτωση του Base Μοντέλου...")
base_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Εφαρμογή των βαρών (LoRA) του Finetuned μοντέλου...")
model = PeftModel.from_pretrained(base_model, extract_path)
model.eval()
print("Το Finetuned Μοντέλο φορτώθηκε επιτυχώς!")

# ── 2. Inference Function ──────────────────────────────────
SYSTEM = "You are an RF expert. Answer with one word only. No explanation."

def predict(image, question):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}
    ]
    text   = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=text, images=image, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,      # greedy decoding
            temperature=1.0,
            pad_token_id=processor.tokenizer.pad_token_id
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer     = processor.tokenizer.decode(new_tokens, skip_special_tokens=True)
    return answer.strip().lower()

# ── 3. Task Definitions ────────────────────────────────────
TASKS = {
    "modulation":  "What is the modulation scheme of this constellation?",
    "phase_noise": "Identify the level of phase noise in this signal.",
    "iq_imbalance":"What is the severity of the I/Q imbalance?",
    "jamming":     "Is there jamming present in this constellation? Describe it.",
    "snr_range":   "Classify the Signal-to-Noise Ratio (SNR) range of this signal.",
}

LABEL_MAP = {
    "none": "none", "mild": "mild", "medium": "medium", "severe": "severe",
    "very_low": "very_low", "low": "low", "high": "high", "very_high": "very_high",
    "bpsk": "bpsk", "qpsk": "qpsk",
    "4-ask": "4-ask", "8-ask": "8-ask",
    "16-qam": "16-qam", "32-qam": "32-qam", "64-qam": "64-qam",
    "128-qam": "128-qam", "256-qam": "256-qam",
    "4-hqam": "4-hqam", "16-hqam": "16-hqam", "64-hqam": "64-hqam",
    "16-apsk": "16-apsk", "32-apsk": "32-apsk",
    "64-apsk": "64-apsk", "128-apsk": "128-apsk",
}

def normalize(text):
    text = text.strip().lower()
    text = text.replace('"', '').replace("'", '').replace('{', '').replace('}', '')
    text = text.split()[0] if text.split() else text
    return LABEL_MAP.get(text, text)

# ── 4. Evaluation ──────────────────────────────────────────
image_labels = defaultdict(dict)
with open("/content/rf_vqa_dataset.jsonl", "r") as f:
    for line in f:
        s = json.loads(line)
        img = s["image"]
        for task, question in TASKS.items():
            if s["question"] == question:
                image_labels[img][task] = s["answer"].lower()

try:
    valid_images = [img for img in image_labels.keys() if img in test_imgs]
except NameError:
    valid_images = list(image_labels.keys())

random.seed(42)
random.shuffle(valid_images)
images_to_eval = valid_images[:2000]

print(f"\nΣύνολο εικόνων στο dataset: {len(image_labels)}")
print(f"Εικόνες που θα αξιολογηθούν με το FINETUNED Μοντέλο: {len(images_to_eval)}\n")

results    = defaultdict(lambda: {"correct": 0, "total": 0})
all_preds  = defaultdict(list)
all_labels = defaultdict(list)

for i, img_path in enumerate(images_to_eval):
    labels = image_labels[img_path]

    loaded_image = Image.open(img_path).convert("RGB")

    for task, question in TASKS.items():
        if task not in labels:
            continue

        true_label = labels[task]
        pred_label = normalize(predict(loaded_image, question))
        true_norm  = normalize(true_label)

        is_correct = pred_label == true_norm
        results[task]["correct"] += int(is_correct)
        results[task]["total"]   += 1
        all_preds[task].append(pred_label)
        all_labels[task].append(true_norm)

    if (i+1) % 50 == 0:
        print(f"  [{i+1}/{len(images_to_eval)}] εικόνες αξιολογήθηκαν...", flush=True)

# ── 5. Results ─────────────────────────────────────────────
print("\n" + "="*55)
print("       ΑΠΟΤΕΛΕΣΜΑΤΑ FINETUNED ΜΟΝΤΕΛΟΥ (2000 ΕΙΚΟΝΕΣ)")
print("="*55)

total_correct = 0
total_samples = 0

for task in TASKS.keys():
    r       = results[task]
    acc     = r["correct"] / r["total"] * 100 if r["total"] > 0 else 0
    total_correct += r["correct"]
    total_samples += r["total"]
    print(f"  {task:<20} {acc:6.2f}%  ({r['correct']}/{r['total']})")

overall = total_correct / total_samples * 100 if total_samples > 0 else 0
print("-"*55)
print(f"  {'ΣΥΝΟΛΟ':<20} {overall:6.2f}%  ({total_correct}/{total_samples})")
print("="*55)

# ── 6. Confusion Matrix ανά task ──────────────────────────
for task in TASKS.keys():
    preds  = all_preds[task]
    labels = all_labels[task]

    if not preds:
        continue

    classes = sorted(set(labels + preds))

    print(f"\n{'='*55}")
    print(f"  Classification Report — {task.upper()}")
    print(f"{'='*55}")
    print(classification_report(labels, preds,
                                 labels=classes,
                                 target_names=classes,
                                 zero_division=0))

    cm  = confusion_matrix(labels, preds, labels=classes)
    fig, ax = plt.subplots(figsize=(max(5, len(classes)), max(4, len(classes)-1)))
    im  = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(classes, fontsize=8)

    thresh = cm.max() / 2
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, str(cm[i,j]),
                    ha='center', va='center', fontsize=9,
                    color='white' if cm[i,j] > thresh else 'black')

    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('True',      fontsize=10)
    ax.set_title(f'FINETUNED MODEL Confusion Matrix — {task}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"/content/finetuned_confusion_{task}.png", dpi=100)
    plt.show()

In [ ]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

local_temp_path = '/content/smolvlm_rf_final_temp'
os.makedirs(local_temp_path, exist_ok=True)

print(" Αποθήκευση του μοντέλου και του processor τοπικά...")
peft_model.save_pretrained(local_temp_path)
processor.save_pretrained(local_temp_path)

drive_save_dir = '/content/drive/MyDrive/MyTrainedModels'
os.makedirs(drive_save_dir, exist_ok=True)

zip_base_name = os.path.join(drive_save_dir, 'smolvlm_rf_final')
print(f" Συμπίεση σε ZIP και μεταφορά στο Drive ({zip_base_name}.zip)...")

shutil.make_archive(zip_base_name, 'zip', local_temp_path)

print(f"ΟΛΟΚΛΗΡΩΘΗΚΕ! Το μοντέλο συμπιέστηκε και αποθηκεύτηκε με ασφάλεια στο: {zip_base_name}.zip")

In [ ]:
!unzip -q "/content/drive/MyDrive/generalization_test.zip" -d "/content/"

In [ ]:
import torch
import sys
import os
import zipfile
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ── 1. Αποσυμπίεση και Φόρτωση Μοντέλου ────────────────────
zip_path = "/content/drive/MyDrive/MyTrainedModels/smolvlm_rf_final.zip"
extract_path = "/content/smolvlm_rf_final"

if not os.path.exists(extract_path):
    print(f"Εξαγωγή του μοντέλου από {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

model_id  = "HuggingFaceTB/SmolVLM-256M-Instruct"
processor = AutoProcessor.from_pretrained(model_id)

print(" Φόρτωση του Base Μοντέλου...")
base_model = AutoModelForImageTextToText.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto"
)

print(" Εφαρμογή των βαρών (LoRA)...")
model = PeftModel.from_pretrained(base_model, extract_path)
model.eval()
print(" Το Finetuned VLM φορτώθηκε επιτυχώς!")

# ── 2. Load Dataset ────────────────────────────────────────
csv_path = "/content/generalization_test/labels.csv"
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

print(f" Dataset: {len(df)} εικόνες φορτώθηκαν (Χωρίς test_category)")

# ── 3. Task Definitions ────────────────────────────────────
SYSTEM = "You are an RF expert. Answer with one word only. No explanation."

TASKS = {
    "modulation":  "What is the modulation scheme of this constellation?",
    "phase_noise": "Identify the level of phase noise in this signal.",
    "iq_imbalance":"What is the severity of the I/Q imbalance?",
    "jamming":     "Is there jamming present in this constellation? Describe it.",
    "snr_range":   "Classify the Signal-to-Noise Ratio (SNR) range of this signal.",
}

TASK_TO_LABEL = {
    "modulation":  "modulation",
    "phase_noise": "phase_noise",
    "iq_imbalance":"iq_imbalance",
    "jamming":     "jamming",
    "snr_range":   "snr_range",
}

# ── 4. Inference Functions ─────────────────────────────────
def vlm_predict(image_path, question):
    image    = Image.open(image_path).convert("RGB")
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}
    ]
    text   = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=text, images=image, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=10, do_sample=False, pad_token_id=processor.tokenizer.pad_token_id
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer     = processor.tokenizer.decode(new_tokens, skip_special_tokens=True)
    return answer.strip().lower().split()[0] if answer.strip() else "unknown"

# ── 5. Evaluation Loop ─────────────────────────────────────
correct_preds = {task: 0 for task in TASKS.keys()}
snr_results = defaultdict(lambda: defaultdict(list))

total_images = len(df)
print(f"\n🚀 Ξεκινάει evaluation σε {total_images} εικόνες...\n")

for i, row in df.iterrows():
    img_path = "/content/generalization_test/images/" + str(row['filename'])
    snr      = row['snr_db']

    if (i+1) % 50 == 0:
        print(f"  [{i+1}/{total_images}] εικόνες αξιολογήθηκαν...", flush=True)
        sys.stdout.flush()

    for task, question in TASKS.items():
        true_label = str(row[TASK_TO_LABEL[task]]).lower()
        pred_label = vlm_predict(img_path, question)

        if "_to_" in true_label:
            acceptable_answers = true_label.split("_to_")
            is_correct = int(pred_label in acceptable_answers)
        else:
            is_correct = int(pred_label == true_label)

        correct_preds[task] += is_correct
        snr_results[task][snr].append(is_correct)
print("\n Evaluation ολοκληρώθηκε!")

print("\n" + "="*50)
print(f" ΑΠΟΤΕΛΕΣΜΑΤΑ SmolVLM (σε {total_images} OOD εικόνες)")
print("="*50)



print(f"  {'Task':<20} | {'SmolVLM %':>10} ")
print("-" * 50)
for task in TASKS.keys():
    vlm_acc = (correct_preds[task] / total_images) * 100
    print(f"  {task:<20} | {vlm_acc:9.2f}% ")
print("="*50)

print("\n📈 Δημιουργία Accuracy vs SNR plots...")

all_snrs = sorted(set(df['snr_db'].unique()))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes      = axes.flatten()

for ax_idx, task in enumerate(TASKS.keys()):
    ax = axes[ax_idx]
    vlm_accs = []

    for snr in all_snrs:
        preds = snr_results[task][snr]
        vlm_accs.append(np.mean(preds) * 100 if preds else 0)

    ax.plot(all_snrs, vlm_accs, 'b-o', linewidth=2, markersize=5, label='SmolVLM-256M')

    unseen_snrs = [3, 7, 12, 22, 27, -5, -2, 40, 45]
    for snr in unseen_snrs:
        if snr in all_snrs:
            ax.axvline(x=snr, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)

    ax.set_title(f'{task.replace("_", " ").title()}', fontsize=12, fontweight='bold')
    ax.set_xlabel('SNR (dB)', fontsize=10)
    ax.set_ylabel('Accuracy (%)', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([-5, 105])
    ax.set_xlim([min(all_snrs)-1, max(all_snrs)+1])

axes[5].axis('off')


plt.suptitle('Accuracy vs SNR — SmolVLM-256M\nGeneralization & Robustness Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/accuracy_vs_snr.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Αποθηκεύτηκε: accuracy_vs_snr.png")